In [26]:
import os
import glob

folder = '/Users/sebastianx/Corpus Linguistics/BNC/Texts'

try:
    from google.colab import files
    uploaded_files = files.upload()
except ImportError:
    uploaded_files = {}
    for file_path in glob.glob(os.path.join(folder, '*')):
        if os.path.isfile(file_path):
            filename = os.path.basename(file_path)
            uploaded_files[filename] = open(file_path, 'rb').read()

print(f"已加载 {len(uploaded_files)} 个文件：")
for name in uploaded_files:
    print(f"  {name}")

已加载 4049 个文件：
  AKK.xml
  CPT.xml
  F8F.xml
  AHP.xml
  CP5.xml
  H5N.xml
  FMM.xml
  JYE.xml
  F9B.xml
  CRK.xml
  H4J.xml
  AHG.xml
  H5Y.xml
  CSX.xml
  H58.xml
  CS9.xml
  CPC.xml
  JXV.xml
  HBM.xml
  HA7.xml
  JY3.xml
  HAV.xml
  AJ9.xml
  AJX.xml
  H7F.xml
  F9U.xml
  ADS.xml
  H9M.xml
  AD2.xml
  JTB.xml
  A2C.xml
  A19.xml
  FAN.xml
  FB4.xml
  A1X.xml
  FBU.xml
  HLF.xml
  A3G.xml
  FC0.xml
  AE6.xml
  ADD.xml
  A25.xml
  JT4.xml
  FBB.xml
  FA8.xml
  A2T.xml
  FAY.xml
  HL0.xml
  A31.xml
  A0K.xml
  FCF.xml
  A3P.xml
  HM4.xml
  HXC.xml
  FUK.xml
  J5L.xml
  AP7.xml
  CHR.xml
  J6W.xml
  APV.xml
  CH3.xml
  J72.xml
  J7S.xml
  CJM.xml
  HYG.xml
  JAC.xml
  HX5.xml
  HXT.xml
  CHE.xml
  J7D.xml
  D96.xml
  FTX.xml
  HY1.xml
  JA5.xml
  HYP.xml
  FT9.xml
  FYH.xml
  CGJ.xml
  CD0.xml
  CE4.xml
  CFN.xml
  J8K.xml
  CEU.xml
  FXL.xml
  HUD.xml
  HTW.xml
  HWL.xml
  HT6.xml
  J9X.xml
  CDF.xml
  J99.xml
  CF8.xml
  CEB.xml
  CFY.xml
  JNL.xml
  HUS.xml
  HU2.xml
  HVH.xml
  KD9.

In [27]:
import xml.etree.ElementTree as ET

In [ ]:
# Create KWIC concordance lines for "help" from our XML files

help_KWIC = []
help_form = []
help_pos = []

max_buffer = 10  # for left context, keep last 10 words

# Metadata lists
file_id = []
genre = []
mode = []
subgenre = []
year = []
hits = []
hit = 1
corpus = []
variety = []

# Speaker data (for Spoken texts only, "NA" for Written)
speaker_id = []
speaker_gender = []
speaker_age = []
soc = []

dep_var = []

# BNC age group codes → midpoint of age range
# Actual codes found in BNC: Ag0 (unknown), Ag1-Ag5, X (unknown)
age_midpoint = {
    'Ag0': 'unknown',
    'Ag1': 7,    # 0-14
    'Ag2': 20,   # 15-24
    'Ag3': 30,   # 25-34
    'Ag4': 40,   # 35-44
    'Ag5': 55,   # 45+
    'X':   'unknown',
}

# XML namespace for xml:id attribute
XML_NS = '{http://www.w3.org/XML/1998/namespace}'

# Helper function: determine DepVar from elements after "help"
# Uses BNC c5 tags: TO0=to-infinitive marker, VVI=bare infinitive, VVG=-ing form
def get_dep_var(elements, help_idx, help_c5):
    if not help_c5 or not help_c5.startswith('VV'):
        return "NA"  # help is not a verb
    word_count = 0
    i = help_idx + 1
    while i < len(elements) and word_count < 15:
        elem = elements[i]
        i += 1
        if elem.tag not in ['w', 'c']:
            continue
        c5 = elem.get('c5', '')
        text = (elem.text or '').strip().lower()
        if elem.tag == 'w':
            word_count += 1
        # Stop at sentence boundaries or new finite clauses
        if text in ['.', '!', '?'] or c5 in ['VVD', 'VVZ']:
            break
        if text in ['and', 'or', 'that', 'if', 'whether']:
            break
        # TO: "to" followed by infinitive (skip adverbs/negation in between)
        if elem.tag == 'w' and (c5 == 'TO0' or text == 'to'):
            for j in range(i, min(i + 5, len(elements))):
                nxt = elements[j]
                if nxt.tag == 'w':
                    if nxt.get('c5', '') in ['VVI', 'VVG', 'VVB']:
                        return "TO"
                    if nxt.get('c5', '') not in ['AV0', 'XX0']:
                        break
        # INING: "in" + -ing
        if elem.tag == 'w' and text == 'in':
            for j in range(i, min(i + 3, len(elements))):
                nxt = elements[j]
                if nxt.tag == 'w':
                    if nxt.get('c5', '') == 'VVG':
                        return "INING"
                    break
        # ING: -ing form directly after help
        if elem.tag == 'w' and c5 == 'VVG':
            return "ING"
        # BARE: bare infinitive directly after help
        if elem.tag == 'w' and c5 in ['VVI', 'VVB']:
            return "BARE"
    return "NA"


for file, content in uploaded_files.items():
    tree = ET.parse(io.BytesIO(content))

    # --- (1) File-level metadata ---

    id = file.replace('.xml', '')

    classcode = tree.find('.//classCode')
    meta = classcode.text

    creation = tree.find('.//creation')
    yearvalue = creation.get('date')
    if yearvalue == '0000':
        yearvalue = '1990'

    # Determine mode and extract mode-specific metadata once per file
    if meta[0] == "W":
        modevalue = "Written"
        wstext_element = tree.find('.//wtext') or tree.find('.//stext')
        genrevalue = wstext_element.get('type') if wstext_element is not None else "NA"
        subgenrevalue = meta[2:]
        spk_id, spk_gender, spk_age, spk_soc = "NA", "NA", "NA", "NA"

    elif meta[0] == "S":
        modevalue = "Spoken"
        genrevalue = "NA"
        subgenrevalue = "NA"
        person = tree.find('.//person')
        if person is not None:
            spk_id = person.get(f'{XML_NS}id', 'NA')   # fix: use XML namespace
            spk_gender = person.get('sex', 'NA')
            raw_age = person.get('ageGroup', 'Ag0')
            spk_age = age_midpoint.get(raw_age, 'unknown')  # fix: Ag1-Ag5 codes
            spk_soc = person.get('soc', 'NA')
        else:
            spk_id, spk_gender, spk_age, spk_soc = "NA", "NA", "unknown", "NA"

    else:
        modevalue = "Unknown"
        wstext_element = tree.find('.//wtext') or tree.find('.//stext')
        genrevalue = wstext_element.get('type') if wstext_element is not None else "NA"
        subgenrevalue = meta[2:]
        spk_id, spk_gender, spk_age, spk_soc = "NA", "NA", "NA", "NA"

    # --- (2) KWIC concordance ---

    left_context = []
    collecting_right = False
    right_context = []
    current_right_context_words = 0

    all_elems = list(tree.iter())

    for elem_idx, elem in enumerate(all_elems):

        # If we're collecting right context (after a "help" hit)
        if collecting_right:
            if elem.tag in ['w', 'c']:
                right_context.append(elem.text)
                if elem.tag == 'w':
                    current_right_context_words += 1
                    if current_right_context_words >= 20:
                        collecting_right = False
                        right_KWIC = ' '.join(filter(None, right_context))
                        help_KWIC.append(f"{left_KWIC} {help_word} {right_KWIC}")

        # Build left context buffer
        if elem.tag in ['w', 'c'] and elem.text and not elem.text.lower().startswith('help'):
            left_context.append(elem.text)
            if len(left_context) > max_buffer:
                left_context.pop(0)

        # Found a "help" hit
        if elem.tag == 'w' and elem.text and elem.text.lower().startswith('help'):

            # If still collecting right context from a previous hit, store it first
            if collecting_right:
                right_KWIC = ' '.join(filter(None, right_context))
                help_KWIC.append(f"{left_KWIC} {help_word} {right_KWIC}")

            left_KWIC = ' '.join(left_context)
            help_word = elem.text
            help_form.append(help_word)
            help_pos.append(elem.get('c5'))

            dep_var.append(get_dep_var(all_elems, elem_idx, elem.get('c5')))

            # Append all metadata for this hit
            file_id.append(id)
            genre.append(genrevalue)
            mode.append(modevalue)
            subgenre.append(subgenrevalue)
            year.append(yearvalue)
            corpus.append("BNC")
            variety.append("BrE")
            speaker_id.append(spk_id)
            speaker_gender.append(spk_gender)
            speaker_age.append(spk_age)
            soc.append(spk_soc)
            hits.append(str(hit))
            hit += 1

            # Start collecting right context
            collecting_right = True
            right_context = []
            current_right_context_words = 0

    # If the last hit's right context wasn't stored yet
    if collecting_right:
        right_KWIC = ' '.join(filter(None, right_context))
        help_KWIC.append(f"{left_KWIC} {help_word} {right_KWIC}")

# --- (3) Build DataFrame and export two Excel files ---

df_all = pd.DataFrame({
    'Hit': hits,
    'KWIC': help_KWIC,
    'Form': help_form,
    'POS': help_pos,
    'DepVar': dep_var,
    'TextID': file_id,
    'Year': year,
    'Genre': genre,
    'Subgenre': subgenre,
    'Mode': mode,
    'Variety': variety,
    'Corpus': corpus,
    'SpeakerID': speaker_id,
    'SpeakerGender': speaker_gender,
    'SpeakerAge': speaker_age,
    'SpeakerSoc': soc
})

# Written: TextID | Year | Genre | Subgenre | Mode | Variety | Corpus
df_written = df_all[df_all['Mode'] == 'Written'][[
    'Hit', 'KWIC', 'Form', 'POS', 'DepVar',
    'TextID', 'Year', 'Genre', 'Subgenre', 'Mode', 'Variety', 'Corpus'
]].reset_index(drop=True)

# Spoken: TextID | SpeakerID | Year | SpeakerGender | SpeakerAge | SpeakerSoc | Mode | Variety | Corpus
df_spoken = df_all[df_all['Mode'] == 'Spoken'][[
    'Hit', 'KWIC', 'Form', 'POS', 'DepVar',
    'TextID', 'SpeakerID', 'Year', 'SpeakerGender', 'SpeakerAge', 'SpeakerSoc', 'Mode', 'Variety', 'Corpus'
]].reset_index(drop=True)

df_written.to_excel('BNC_help_written.xlsx', index=False)
df_spoken.to_excel('BNC_help_spoken.xlsx', index=False)

print(f"Written: {len(df_written)} results → BNC_help_written.xlsx")
print(f"Spoken:  {len(df_spoken)} results → BNC_help_spoken.xlsx")


/var/folders/yt/4l1xrk6s3bz2t7j232zb03fw0000gn/T/ipykernel_23492/958000500.py:108: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  wstext_element = tree.find('.//wtext') or tree.find('.//stext')


Written: 52761 results → BNC_help_written.xlsx
Spoken:  4723 results → BNC_help_spoken.xlsx
